This notebook is the same as `01_make_dataset.ipynb` except that instead of predicting the logprob on 0 or 1, it predicts the logprobs on a chosen or rejected completion

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from loguru import logger
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from einops import rearrange, repeat
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.data import DataCollatorForLanguageModeling

import torch
from torch import Tensor
from torch.nn.functional import (
    binary_cross_entropy_with_logits as bce_with_logits,
)
from torch.nn.functional import (
    cross_entropy,
)
from pathlib import Path
from jaxtyping import Float
from torch import Tensor

import functools
import pandas as pd
import numpy as np

import itertools
from tqdm.auto import tqdm
import random
import json
from tqdm.auto import tqdm
from datasets import concatenate_datasets

from activation_store.collect import activation_store, default_postprocess_result

In [ ]:
import gc
def clear_mem():
    """
    Clear memory
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    return None
clear_mem()

## Load model

In [ ]:
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Qwen/Qwen3-1.7
# Qwen/Qwen3-0.6B-FP8
model_name = "Qwen/Qwen3-4B"
batch_size = 6

model_name = "Qwen/Qwen3-1.7B"
batch_size = 10
# model_name = "Qwen/Qwen3-8B"

# model_name = "unsloth/Llama-3.2-1B-Instruct"

# model_name = "Qwen/Qwen2.5-3B-Instruct"
# model_name = "Qwen/Qwen2.5-3B-Instruct-AWQ"

# model_name = "AMead10/Llama-3.2-3B-Instruct-AWQ"

# model_name = "unsloth/Phi-4-mini-instruct" # 4b
# model_name = "stelterlab/phi-4-AWQ"



In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if ('awq' not in model_name.lower()) else torch.float16,
    device_map="auto",
    attn_implementation="eager",  # flex_attention  flash_attention_2 sdpa eager
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

## Load data and tokenize

In [ ]:
# N = 316
max_length = 128
max_samples = 50
split = "train"
ds1 = load_dataset(path="wassname/genies_preferences", split=split, name='truthful_qa', keep_in_memory=False)

if max_samples is not None:
    ds1 = ds1.shuffle(seed=42).select(range(max_samples))

In [ ]:



def preprocess_activation_ds_rows(rows2):

    # Dict[list] -> list[Dict[str, str]]
    rows = [{k: v[i] for k, v in rows2.items()} for i in range(len(rows2['prompt']))]

    outs = []
    for row in rows:
        # prompt mask
        k = 'chosen'
        sys, q = row['prompt'].split('## Instruction:')
        q = q.split('## Response:')[0]
        messages =[
                {"role": "system", "content": sys},
                {"role": "user", "content": q},
                # {"role": "assistant", "content": row[k]},
            ]
        o =  tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            add_generation_prompt=True,
            padding_side="left",
            truncation_side="left",
        )
        prompt_length = len(o['input_ids'])
        
        for k in ['chosen', 'rejected']:  
            row2 = row.copy()          
            row2['messages'] =[
                    {"role": "system", "content": sys},
                    {"role": "user", "content": q},
                    {"role": "assistant", "content": row[k]},
                ]
            
            o =  tokenizer.apply_chat_template(
                row2['messages'],
                tokenize=True,
                return_dict=True,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                add_generation_prompt=False,
                padding_side="left",
                truncation_side="left",
            )

            whole_length = len(o['input_ids'])
            o['prompt_mask'] = [1] * prompt_length + [0] * (whole_length - prompt_length)


            o['labels'] = o['input_ids'][1:]

            # o = {f"{k}_{kk}": vv for kk, vv in o.items()}
            row2.update(o)
            row2['completion_type'] = k
            outs.append(row2)


    # List[Dict[str, str]] -> Dict[list]
    outs = {k: [row[k] for row in outs] for k in outs[0].keys()}
    return outs


ds2 = ds1.map(preprocess_activation_ds_rows, batched=True).with_format("torch")
ds2

In [ ]:
tokenizer.batch_decode(ds2['input_ids'])[0]

## Data loader

In [ ]:
collate_fn = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

ds3 = ds2.select_columns(
    ['input_ids', 'attention_mask', 'prompt_mask']
).with_format("torch")

dl = DataLoader(ds3, batch_size=batch_size, collate_fn=collate_fn)
print(dl), ds3

## Collect activations

In [ ]:
# # choose layers to cache
# n_layers = model.config.num_hidden_layers
# a = int(0.3*n_layers)
# b = n_layers-2
# layer_groups = {
#     'mlp.down_proj': [k for k,v in model.named_modules() if k.endswith('mlp.down_proj')][a:b],
#     'self_attn': [k for k,v in model.named_modules() if k.endswith('.self_attn')][a:b],
#     'mlp.up_proj': [k for k,v in model.named_modules() if k.endswith('mlp.up_proj')][a:b],
# }
# layer_groups

In [ ]:
# choose layers to cache
n_layers = model.config.num_hidden_layers
a = int(0.5*n_layers)
b = n_layers-2
select = slice(a, b, 1)
layer_groups = {
    'mlp.down_proj': [k for k,v in model.named_modules() if k.endswith('mlp.down_proj')][select],
    'self_attn': [k for k,v in model.named_modules() if k.endswith('.self_attn')][select],
    'mlp.up_proj': [k for k,v in model.named_modules() if k.endswith('mlp.up_proj')][select],
}
layer_groups

In [ ]:
import os, unicodedata, string
from pathlib import Path

def sanitize_path(path: Path | str, allow_period: bool = True) -> Path:
    """
    Whitelist only ASCII letters, digits, dash, underscore,
    optionally period, and forward‐slash. Replace others with '_'.
    """
    s = unicodedata.normalize("NFKD", str(path))\
                     .encode("ascii", "ignore")\
                     .decode()
    s = s.replace(os.sep, "/")
    allowed = set(string.ascii_letters + string.digits + "_-")
    if allow_period: allowed.add(".")
    allowed.add("/")
    return Path("".join(ch if ch in allowed else "_" for ch in s))

In [ ]:

acts_outfile = Path(f'/tmp/activation_store/ds_at-{model_name.replace("/", "")}-truthfulQA-bool-{split}-{len(ds2)}-{max_length}_v4g_wgen.parquet')
acts_outfile = sanitize_path(acts_outfile)
acts_outfile

In [ ]:
def collect_all_tokens(*args, **kwargs):
    # TODO apply prompt mask... only keep tokens after prompt mask
    o = default_postprocess_result(*args, **kwargs, last_token=False)
    B, L, T, H = o['hidden_states'].shape
    for k in o:
        v = o[k]
        ts = slice(None, None)  # last 10 tokens
        if not isinstance(v, Tensor):
            continue
        if v.ndim < 2:
            # print(f"Skipping {k} with shape {v.shape} and dtype {v.dtype}. Not a tensor or has less than 2 dimensions.")
            continue
        if 'mask' in k:
            # skip masks
            continue
        if (v.ndim == 4) and (v.shape[2] == T):
            # remove prompt mask
            pm = args[0]['prompt_mask'].bool() # [B, T]
            # print(pm.shape, k, v.shape,)
            pm = repeat(pm, 'b t -> b 1 t 1')
            o[k] = (v*pm)[:, ts]
        elif (v.ndim == 3) and (v.shape[1] == T):
            # remove prompt mask
            pm = args[0]['prompt_mask'].bool() # [B, T]
            # print(pm.shape, k, v.shape,)
            pm = repeat(pm, 'b t -> b t 1')
            o[k] = (v*pm)[:, ts]
        elif (v.ndim == 2) and (v.shape[1] == T):
            # remove prompt mask
            pm = args[0]['prompt_mask'].bool()
            # print(pm.shape, k, v.shape,)
            o[k] = (v*pm)[:, ts]
        else:
            raise ValueError(f"Unexpected shape {v.shape} for {k}. Expected 2D, 3D or 4D tensor with second dimension equal to T={T}.")

    return o


f = activation_store(dl, model, layers=layer_groups, postprocess_result=collect_all_tokens, 
                     outfile=acts_outfile)
f

In [ ]:
# TODO which is better for mem, this or below?
ds_a = load_dataset("parquet", split='train', data_files=str(f), keep_in_memory=False).with_format("torch")
ds_a

In [ ]:
ds2

In [ ]:
a = ds2.select_columns(
    ['input_ids',  'prompt_mask', 'i', 'completion_type', 'labels', ]
).with_format("torch")
ds_a2a = concatenate_datasets([ds_a, a], axis=1).with_format("torch")
ds_a2a

In [ ]:
act_groups = [c for c in ds_a2a.column_names if c.startswith('acts-')]
act_groups

In [ ]:
for k,v in ds_a2a[:2].items():
    if hasattr(v, 'shape'):
        print(k, v.shape)
    else:
        print(k, type(v))

In [ ]:
# sanity test generate
b = next(iter(dl))
b = {k: v.to(model.device) for k, v in b.items()}
o = model.generate(
    inputs=b["input_ids"],
    attention_mask=b["attention_mask"],
    max_new_tokens=10,
)
gent = tokenizer.batch_decode(o, skip_special_tokens=False)
for g in gent:
    print(g)
    print("---")
    break

## Get supressed activations

In [ ]:

import torch.nn.functional as F

@torch.no_grad()
def get_supressed_activations(
    hs: Float[Tensor, "l b t h"], w_out, w_inv
) -> Float[Tensor, "l b t h"]:
    """
    Novel experiment: Here we define a transform to isolate supressed activations, where we hypothesis that style/concepts/scratchpads and other internal only representations must be stored.

    See the following references for more information:

    - https://arxiv.org/pdf/2401.12181
        - > Suppression neurons that are similar, except decrease the probability of a group of related tokens
        - > We find a striking pattern which is remarkably consistent across the different seeds: after about the halfway point in the model, prediction neurons become increasingly prevalent until the very end of the network where there is a sudden shift towards a much larger number of suppression neurons.

    - https://arxiv.org/html/2406.19384
        - > Previous work suggests that networks contain ensembles of “prediction" neurons, which act as probability promoters [66, 24, 32] and work in tandem with suppression neurons (Section 5.4).


    Output:
    - supression amount: This is a tensor of the same shape as the input hs, where the values are the amount of suppression that occured at that layer, and the sign indicates if it was supressed or promoted. How do we calulate this? We project the hs using the output_projection, look at the diff from the last layer, and then project it back using the inverse of the output projection. This gives us the amount of suppression that occured at that layer.
    """
    hs_flat = rearrange(hs[:, :, :], "l b t h -> (l b t) h")
    logits_flat = F.linear(hs_flat, w_out)
    logits = rearrange(
        logits_flat, "(l b t) h -> l b t h", l=hs.shape[0], b=hs.shape[1], t=hs.shape[2]
    )
    logit_diffs = logits[:, :, :].diff(dim=0)
    logit_diffs_flat = rearrange(logit_diffs, "l b t h -> (l b t) h")
    # W_inv = get_cache_inv(w_out)

    # get the supression projected back
    hs_supr_flat = F.linear(logit_diffs_flat.to(dtype=w_inv.dtype), w_inv)
    hs_inv_flat = F.linear(logits_flat.to(dtype=w_inv.dtype), w_inv)
    supr_amounts = rearrange(
        hs_supr_flat, "(l b t) h -> l b t h", l=hs.shape[0] - 1, b=hs.shape[1], t=hs.shape[2]
    ).to(w_out.dtype)
    hs_inv = rearrange(
        hs_inv_flat, "(l b t) h -> l b t h", l=hs.shape[0], b=hs.shape[1], t=hs.shape[2]
    ).to(w_out.dtype)

    # add on missing first layer
    # torch.zeros_like(supr_amounts[:1]).to(hs.device)
    supr_amounts = torch.cat(
        [torch.zeros_like(supr_amounts[:1]).to(hs.device), supr_amounts], dim=0
    )

    residual = hs - hs_inv # lost in projection
    return supr_amounts, residual

In [ ]:
# def get_uniq_token_ids(tokens):
#     token_ids = tokenizer(
#         tokens, add_special_tokens=False, padding=False
#     ).input_ids
#     token_ids = torch.tensor(list(set([x[0] for x in token_ids]))).long()
#     print("before", tokens)
#     print("after", tokenizer.batch_decode(token_ids))
#     return token_ids


# false_tokens = ["0", "0 ", "0\n", "false", "False "]
# false_token_ids = get_uniq_token_ids(false_tokens)

# true_tokens = ["1", "1 ", "1\n", "true", "True "]
# true_token_ids = get_uniq_token_ids(true_tokens)

# print('QC: manually check that these are equivilent (no <end_of_text> or newline)')

In [ ]:
# now we map to 1) calc supressed activations 2) llm answer (prob of 0 vs prob of 1)

Wo = model.get_output_embeddings().weight.detach().clone().cpu()
Wo_inv = torch.pinverse(Wo.clone().float())


def postprocess_activation_ds_rows(o):
    """Process model outputs"""

    # Gather the log probabilities for the actual labels, which is the next token
    labels = o["labels"]#[:, 1:].clone()
    labels = o["input_ids"][1:].clone()
    logits = o["logits"][:-1]
    # print("logits", logits.shape, logits.dtype)
    log_probs = logits.log_softmax(dim=-1)
    o['label_logp'] = torch.gather(
        input=log_probs, dim=-1, index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # get supressed activations
    hs = o["hidden_states"][None]
    hs = rearrange(hs, "b l t h -> l b t h")
    supr_amounts, residual = get_supressed_activations(hs, Wo.to(hs.dtype), Wo_inv.to(hs.dtype))

    # we will only take the last half of layers, and the last token
    layer_half = hs.shape[0] // 2
    
    hs = rearrange(hs, "l b t h -> b l t h").squeeze(0)[layer_half:-2]
    supr_amounts = rearrange(supr_amounts, "l b t h -> b l t h").squeeze(0)[layer_half:-2]
    residual = rearrange(residual, "l b t h -> b l t h").squeeze(0)[layer_half:-2]

    for k in o.keys():
        if k.startswith("acts-"):
            o[k] = o[k][-1:]

    o["hidden_states"] = hs.half()[-1:]
    o["supr_amounts"] = supr_amounts.half()
    o['logits'] = o['logits'].half()
    o['label_logp'] = o['label_logp'].half()
    o['residual'] = residual.half()
    return o


ds_a2 = ds_a2a.map(postprocess_activation_ds_rows, writer_batch_size=1, num_proc=None)
ds_a2

In [ ]:
model = Wo = Wo_inv = tokenizer = None
clear_mem()

In [ ]:



ds = ds_a2 # concatenate_datasets([ds_a2, ds2.remove_columns(['messages', 'attention_mask', 'prompt', 'chosen', 'rejected', 'chosen_messages'])]).with_format("torch")
ds

In [ ]:
{k: v.shape for k,v in ds[:2].items() if isinstance(v, torch.Tensor)}


In [ ]:
out_dir = Path('../data/activation_store')
name = acts_outfile.with_suffix("").relative_to('/tmp/activation_store')
acts_outfile2 = out_dir / name
acts_outfile2.parent.mkdir(parents=True, exist_ok=True)
acts_outfile2

In [ ]:
from datasets import concatenate_datasets, load_dataset


ds_out = ds.with_format("torch")
ds_out.save_to_disk(acts_outfile2)
acts_outfile2

In [ ]:
f_config = acts_outfile2.with_suffix(".json")
json.dump({
    "model_name": model_name,
    "batch_size": batch_size,
    "max_length": max_length,
    'layer_groups': layer_groups,
    # 'model_config': model.config.to_dict(),
    "n_rows": len(ds_out),
}, open(f_config, "w"))
f_config